In [2]:
# -------------------------------------------------------------------------
# CÉLULA 1: Instalar Pacotes e Carregar Base Limpa de BH
# -------------------------------------------------------------------------
!pip install h3 gdown xgboost lightgbm -q

import os
import pandas as pd
import numpy as np
import h3
import warnings
warnings.filterwarnings('ignore')

nome_arquivo = 'renaest_bh_limpo.csv'
caminho_drive = '/content/drive/MyDrive/renaest_bh_limpo.csv'

# 1. Verifica se o arquivo está na sessão local do Colab
if os.path.exists(nome_arquivo):
    df_bh = pd.read_csv(nome_arquivo, sep=';', encoding='utf-8')
    print(f"Base carregada da sessão local: {len(df_bh):,} registros.")

# 2. Caso contrário, monta o Google Drive e busca do Drive
else:
    print("Arquivo local não encontrado. Montando Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

    if os.path.exists(caminho_drive):
        df_bh = pd.read_csv(caminho_drive, sep=';', encoding='utf-8')
        print(f"Base carregada com sucesso do Google Drive: {len(df_bh):,} registros.")
    else:
        print("⚠️ ERRO: O arquivo 'renaest_bh_limpo.csv' não foi encontrado na raiz do Drive.")
        print("Garanta que executou a exportação ao final da Sprint 1 ou faça o upload manual pelo painel lateral esquerdo do Colab.")

Arquivo local não encontrado. Montando Google Drive...
Mounted at /content/drive
Base carregada com sucesso do Google Drive: 476,428 registros.


In [3]:
# -------------------------------------------------------------------------
# CÉLULA 2: Pré-processamento e Garantia dos Tipos de Dados
# -------------------------------------------------------------------------

# 1. Garantir conversão numérica das coordenadas geográficas
df_bh['latitude_acidente'] = pd.to_numeric(df_bh['latitude_acidente'], errors='coerce')
df_bh['longitude_acidente'] = pd.to_numeric(df_bh['longitude_acidente'], errors='coerce')

# 2. Descartar registros sem coordenadas válidas (segurança extra)
df_bh = df_bh.dropna(subset=['latitude_acidente', 'longitude_acidente']).copy()

# 3. Conversão padronizada de datas
df_bh['data_acidente'] = pd.to_datetime(df_bh['data_acidente'], errors='coerce')
df_bh = df_bh.dropna(subset=['data_acidente']).copy()

print(f"Registros validados e prontos para indexação espacial: {len(df_bh):,}")

Registros validados e prontos para indexação espacial: 476,428


In [4]:
# -------------------------------------------------------------------------
# CÉLULA 3: Mapeamento Espacial em Hexágonos Uber H3
# -------------------------------------------------------------------------

# Função para converter Latitude/Longitude em código H3 (compatível com h3-py v3 e v4)
def lat_lon_para_h3(row, resolucao):
    try:
        lat = float(row['latitude_acidente'])
        lon = float(row['longitude_acidente'])
        if hasattr(h3, 'latlng_to_cell'):
            return h3.latlng_to_cell(lat, lon, resolucao)
        return h3.geo_to_h3(lat, lon, resolucao)
    except:
        return None

# Resolução 8: ~0.73 km² por hexágono (Nível de região/bairro)
df_bh['h3_res8'] = df_bh.apply(lambda r: lat_lon_para_h3(r, resolucao=8), axis=1)

# Resolução 9: ~0.10 km² por hexágono (Nível de cruzamento/via)
df_bh['h3_res9'] = df_bh.apply(lambda r: lat_lon_para_h3(r, resolucao=9), axis=1)

# Filtrar qualquer valor nulo gerado no geoprocessamento
df_bh = df_bh.dropna(subset=['h3_res8', 'h3_res9']).copy()

print("Mapeamento H3 concluído com sucesso!")
print("\nAmostra dos hexágonos associados a cada acidente:")
print(df_bh[['latitude_acidente', 'longitude_acidente', 'h3_res8', 'h3_res9']].head())

Mapeamento H3 concluído com sucesso!

Amostra dos hexágonos associados a cada acidente:
   latitude_acidente  longitude_acidente          h3_res8          h3_res9
0         -19.787416          -43.929361  88a88ccb0dfffff  89a88ccb0d3ffff
1         -19.907566          -43.961459  88a88cd869fffff  89a88cd868bffff
2         -19.901353          -43.905328  88a88cd945fffff  89a88cd944bffff
3         -19.932521          -44.035167  88a88cda1bfffff  89a88cda1afffff
4         -19.979071          -43.984213  88a881a591fffff  89a881a5903ffff


In [5]:
# -------------------------------------------------------------------------
# CÉLULA 4: Engenharia de Atributos Espaço-Temporais e Gravidade
# -------------------------------------------------------------------------

# 1. Atributos Calendários
df_bh['ano'] = df_bh['data_acidente'].dt.year
df_bh['mes'] = df_bh['data_acidente'].dt.month
df_bh['dia_semana_num'] = df_bh['data_acidente'].dt.dayofweek
df_bh['e_final_semana'] = df_bh['dia_semana_num'].isin([5, 6]).astype(int)

# 2. Extração da hora e criação de faixas operacionais
df_bh['hora_num'] = pd.to_numeric(
    df_bh['hora_acidente'].astype(str).str.split(':').str[0],
    errors='coerce'
).fillna(-1)

df_bh['horario_pico'] = df_bh['hora_num'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df_bh['periodo_noturno'] = df_bh['hora_num'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)

# 3. Tratamento e criação do Índice de Severidade
df_bh['qtde_obitos'] = pd.to_numeric(df_bh['qtde_obitos'], errors='coerce').fillna(0)
df_bh['qtde_feridosilesos'] = pd.to_numeric(df_bh['qtde_feridosilesos'], errors='coerce').fillna(0)
df_bh['qtde_envolvidos'] = pd.to_numeric(df_bh['qtde_envolvidos'], errors='coerce').fillna(0)

# Flag de Acidente Grave (Presença de vítima lesionada ou fatal)
df_bh['acidente_grave'] = ((df_bh['qtde_obitos'] > 0) | (df_bh['qtde_feridosilesos'] > 0)).astype(int)

print("Novas variáveis temporais e de severidade geradas:")
print(df_bh[['ano', 'mes', 'e_final_semana', 'horario_pico', 'periodo_noturno', 'acidente_grave']].head())

Novas variáveis temporais e de severidade geradas:
    ano  mes  e_final_semana  horario_pico  periodo_noturno  acidente_grave
0  2018    1               0             0                0               0
1  2018    1               0             0                0               1
2  2018    1               0             0                0               0
3  2018    2               0             0                0               1
4  2018    2               0             0                0               1


In [6]:
# -------------------------------------------------------------------------
# CÉLULA 5: Agrupamento da Grade H3 e Construção do Target ('is_hotspot')
# -------------------------------------------------------------------------

# Agrupamento da base por Célula H3 (Resolução 8) e Mês/Ano
df_grid = df_bh.groupby(['h3_res8', 'ano', 'mes']).agg(
    total_acidentes=('num_acidente', 'count'),
    total_graves=('acidente_grave', 'sum'),
    total_obitos=('qtde_obitos', 'sum'),
    ocorrencias_pico=('horario_pico', 'sum'),
    ocorrencias_noturnas=('periodo_noturno', 'sum'),
    ocorrencias_fds=('e_final_semana', 'sum')
).reset_index()

# Definição do Target Preditivo: Células com contagem mensal >= Percentil 75 são Hotspots
limiar_p75 = df_grid['total_acidentes'].quantile(0.75)
df_grid['is_hotspot'] = (df_grid['total_acidentes'] >= limiar_p75).astype(int)

print(f"Limiar para classificação de Hotspot (75º Percentil): >= {limiar_p75:.0f} acidentes/mês no hexágono")
print("\nDistribuição do Target 'is_hotspot':")
print(df_grid['is_hotspot'].value_counts(normalize=True).rename({0: 'Baixo Risco (0)', 1: 'Hotspot (1)'}))
print(f"\nTotal de amostras na grade H3 prontas para treinamento: {len(df_grid):,}")

Limiar para classificação de Hotspot (75º Percentil): >= 17 acidentes/mês no hexágono

Distribuição do Target 'is_hotspot':
is_hotspot
Baixo Risco (0)    0.749605
Hotspot (1)        0.250395
Name: proportion, dtype: float64

Total de amostras na grade H3 prontas para treinamento: 38,559


In [7]:
# -------------------------------------------------------------------------
# CÉLULA 6: Divisão Treino/Teste e Normalização dos Dados
# -------------------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Seleção das Features Explicativas e da Variável Alvo
features = ['mes', 'total_graves', 'total_obitos', 'ocorrencias_pico', 'ocorrencias_noturnas', 'ocorrencias_fds']
X = df_grid[features]
y = df_grid['is_hotspot']

# 2. Divisão com Estratificação (Preserva a proporção de hotspots)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 3. Normalização das Variáveis
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Divisão concluída:")
print(f"- Amostras de Treino: {X_train.shape[0]:,}")
print(f"- Amostras de Teste: {X_test.shape[0]:,}")

Divisão concluída:
- Amostras de Treino: 28,919
- Amostras de Teste: 9,640


In [8]:
# -------------------------------------------------------------------------
# CÉLULA 7: Treinamento e Avaliação Comparativa dos Modelos
# -------------------------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Dicionário com os algoritmos do MVP
modelos = {
    "Regressão Logística": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

resultados = []

for nome, modelo in modelos.items():
    # Modelos lineares utilizam dados escalados; modelos de árvore utilizam dados brutos
    X_tr = X_train_scaled if nome == "Regressão Logística" else X_train
    X_te = X_test_scaled if nome == "Regressão Logística" else X_test

    # Fit e Predição
    modelo.fit(X_tr, y_train)
    y_pred = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:, 1] if hasattr(modelo, "predict_proba") else y_pred

    # Cálculo de Métricas
    resultados.append({
        "Modelo": nome,
        "Acurácia": round(accuracy_score(y_test, y_pred), 4),
        "Precisão": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Revocação (Recall)": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1-Score": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4)
    })

# Tabela comparativa ordenada pelo ROC-AUC
df_comparativo = pd.DataFrame(resultados).sort_values(by="ROC-AUC", ascending=False)

print("=== TABELA COMPARATIVA DE DESEMPENHO (BASELINE) ===")
print(df_comparativo.to_string(index=False))

=== TABELA COMPARATIVA DE DESEMPENHO (BASELINE) ===
             Modelo  Acurácia  Precisão  Revocação (Recall)  F1-Score  ROC-AUC
Regressão Logística    0.9957    1.0000              0.9830    0.9914   0.9998
            XGBoost    0.9944    0.9933              0.9843    0.9888   0.9998
           LightGBM    0.9949    0.9954              0.9843    0.9898   0.9997
      Random Forest    0.9944    0.9937              0.9838    0.9888   0.9981


In [9]:
# -------------------------------------------------------------------------
# CÉLULA 8: Exportar Modelo Campeão e Scaler
# -------------------------------------------------------------------------
import joblib

# Identifica o melhor modelo baseado no ROC-AUC
melhor_nome = df_comparativo.iloc[0]['Modelo']
melhor_modelo = modelos[melhor_nome]

# Nomes dos arquivos de saída
arq_modelo = 'modelo_hotspot_bh_h3.joblib'
arq_scaler = 'scaler_bh_h3.joblib'

# Exportar localmente
joblib.dump(melhor_modelo, arq_modelo)
joblib.dump(scaler, arq_scaler)

# Tentar salvar também no Google Drive para persistência
try:
    joblib.dump(melhor_modelo, f'/content/drive/MyDrive/{arq_modelo}')
    joblib.dump(scaler, f'/content/drive/MyDrive/{arq_scaler}')
    print("Cópias do modelo e scaler salvas no Google Drive com sucesso!")
except:
    pass

print(f"\nModelo campeão ('{melhor_nome}') e scaler exportados como '{arq_modelo}' e '{arq_scaler}'.")

Cópias do modelo e scaler salvas no Google Drive com sucesso!

Modelo campeão ('Regressão Logística') e scaler exportados como 'modelo_hotspot_bh_h3.joblib' e 'scaler_bh_h3.joblib'.
